# 第51章 成对关系图（pairplot）

用pairplot和PairGrid快速筛查多个数值变量的成对关系。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

探索阶段同时检查少量数值变量的分布、相关和分组结构。

## 数据结构

多个数值列，可增加一列分类hue；建议先抽样。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 corner=True 改为 corner=False，观察完整矩阵与下三角矩阵的信息冗余度
2. 修改 diag_kind="hist" 为 diag_kind="kde"，对比直方图与密度图在对角线的显示效果
3. 调整 plot_kws 中的 alpha 参数（如 0.3 或 0.7），说明透明度对多变量散点图的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category=diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value=diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend=taxis["tip"], sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date=pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region="AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
sample = marketing.sample(150, random_state=51)
grid = sns.pairplot(sample, vars=["visits", "ad_spend", "sales", "conversion"], corner=True, diag_kind="hist", plot_kws={"alpha": 0.45, "s": 22})
grid.fig.suptitle("营销指标成对关系", y=1.02)
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
sample = marketing.sample(160, random_state=510)
grid = sns.pairplot(sample, vars=["visits", "ad_spend", "sales"], hue="channel", corner=True, diag_kind="hist", palette="colorblind", plot_kws={"alpha": 0.5, "s": 24})
grid.fig.suptitle("分渠道营销指标关系", y=1.02)
plt.show()


## 3. 参数说明

- vars：选择变量
- corner：下三角
- diag_kind：对角图
- plot_kws：点样式


## 4. 结果解读

沿对角线看单变量分布，非对角线看成对关系；再选择重点关系制作最终图。


## 常见误区

- 变量过多产生巨大矩阵
- 大样本不抽样
- 把探索矩阵直接作为最终报告


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
sample = orders.sample(140, random_state=511)
grid = sns.pairplot(sample, vars=["order_value", "items"], hue="category", diag_kind="hist", palette="Set2", plot_kws={"alpha": 0.55, "s": 25})
grid.fig.suptitle("订单指标与品类", y=1.02)
plt.show()


## 本章小结

用pairplot和PairGrid快速筛查多个数值变量的成对关系。


### 你已经掌握

- 判断成对关系图（pairplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 探索阶段同时检查少量数值变量的分布、相关和分组结构。 |
| 数据结构 | 多个数值列，可增加一列分类hue；建议先抽样。 |
| 结果解读 | 沿对角线看单变量分布，非对角线看成对关系；再选择重点关系制作最终图。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `vars` | 选择变量 |
| `corner` | 下三角 |
| `diag_kind` | 对角图 |
| `plot_kws` | 点样式 |


### 需要注意

- 变量过多产生巨大矩阵
- 大样本不抽样
- 把探索矩阵直接作为最终报告


### 完成检查

- [ ] 能判断什么问题适合使用成对关系图（pairplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
